In this notebook, we'll perform feature engineering and data preparation. We'll clean, merge, and enrich the datasets with population, surface area, and vehicle fleet data to create the final indicators for the subsequent analysis.

In [ ]:
import pandas as pd

In [2]:
df_incidents = pd.read_csv("data/traffic_incidents.csv")
df_municipalities = pd.read_csv("data/municipalities_metrics.csv")
df_aci = pd.read_csv("data/aci_fleet.csv")

In [7]:
#a quick check 
df_incidents.info()

<class 'pandas.DataFrame'>
RangeIndex: 257715 entries, 0 to 257714
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   REF_AREA        257715 non-null  int64
 1   DATA_TYPE       257715 non-null  str  
 2   RESULT          257715 non-null  str  
 3   TIME_PERIOD     257715 non-null  int64
 4   OBS_VALUE       257715 non-null  int64
 5   Codice_Regione  257715 non-null  int64
 6   Comune          257682 non-null  str  
 7   Regione         257715 non-null  str  
dtypes: int64(4), str(4)
memory usage: 15.7 MB


In [ ]:
#After checking the data, we discovered that the municipality with code 1168 is a town named None. This specific name caused an anomaly as Pandas parsed the string "None" as a missing value (NaN). 
#We will add the keep_default_na=False argoument.
df_incidents[df_incidents["Comune"].isna()]

#IN TEORIA GIA SITEMATO; CONTROLLA

,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,Codice_Regione,Comune,Regione
5325,1168,KILLINJ,F,2014,15,1,NaN,Piemonte
5326,1168,KILLINJ,F,2015,14,1,NaN,Piemonte
5327,1168,KILLINJ,F,2016,14,1,NaN,Piemonte
5328,1168,KILLINJ,F,2017,25,1,NaN,Piemonte
5329,1168,KILLINJ,F,2018,14,1,NaN,Piemonte
5330,1168,KILLINJ,F,2019,11,1,NaN,Piemonte
5331,1168,KILLINJ,F,2020,11,1,NaN,Piemonte
5332,1168,KILLINJ,F,2021,11,1,NaN,Piemonte
5333,1168,KILLINJ,F,2022,10,1,NaN,Piemonte
5334,1168,KILLINJ,F,2023,9,1,NaN,Piemonte


In [11]:
df_incidents = pd.read_csv("data/traffic_incidents.csv", keep_default_na=False)
df_incidents.info()

<class 'pandas.DataFrame'>
RangeIndex: 257715 entries, 0 to 257714
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   REF_AREA        257715 non-null  int64
 1   DATA_TYPE       257715 non-null  str  
 2   RESULT          257715 non-null  str  
 3   TIME_PERIOD     257715 non-null  int64
 4   OBS_VALUE       257715 non-null  int64
 5   Codice_Regione  257715 non-null  int64
 6   Comune          257715 non-null  str  
 7   Regione         257715 non-null  str  
dtypes: int64(4), str(4)
memory usage: 15.7 MB


In [14]:
df_municipalities = pd.read_csv("data/municipalities_metrics.csv", keep_default_na=False)
df_municipalities.info()

<class 'pandas.DataFrame'>
RangeIndex: 7896 entries, 0 to 7895
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Codice_Regione      7896 non-null   int64  
 1   Codice_Comune       7896 non-null   int64  
 2   Comune              7896 non-null   str    
 3   Popolazione_Legale  7896 non-null   int64  
 4   Superficie(Kmq)     7896 non-null   float64
 5   Regione             7896 non-null   str    
dtypes: float64(1), int64(3), str(2)
memory usage: 370.3 KB


In [17]:
df_aci

,Regione,Parco_Auto_2024
0,Piemonte,3.075.162
1,Valle D'Aosta,237.162
2,Lombardia,6.426.326
3,Trentino A.A.,1.291.248
4,Veneto,3.302.750
5,Friuli V.G.,828.929
6,Liguria,850.098
7,Emilia Romagna,3.082.830
8,Toscana,2.722.906
9,Umbria,659.203


In [21]:
#we can change the datatype for the "Parco_Auto_2024" columns into int64
df_aci["Parco_Auto_2024"] = df_aci["Parco_Auto_2024"].astype(str).str.replace(".", "").astype(int)
df_aci.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Regione          20 non-null     str  
 1   Parco_Auto_2024  20 non-null     int64
dtypes: int64(1), str(1)
memory usage: 452.0 bytes


We can now proceed with further dataset manipulation, trying to create three distinct columns for the "DATA_TYPE" column

In [49]:
df_acc = df_incidents[(df_incidents["DATA_TYPE"] == "ROADACC") & (df_incidents["RESULT"] == "9")][["REF_AREA", "TIME_PERIOD", "OBS_VALUE", "Codice_Regione", "Comune", "Regione"]].rename(columns={"OBS_VALUE":"Incidenti"}).reset_index(drop=True)
df_inj = df_incidents[(df_incidents["DATA_TYPE"] == "KILLINJ") & (df_incidents["RESULT"] == "F")][["REF_AREA", "TIME_PERIOD", "OBS_VALUE", "Codice_Regione", "Comune", "Regione"]].rename(columns={"OBS_VALUE":"Feriti"}).reset_index(drop=True)
df_dead = df_incidents[(df_incidents["DATA_TYPE"] == "KILLINJ") & (df_incidents["RESULT"] == "M")][["REF_AREA", "TIME_PERIOD", "OBS_VALUE", "Codice_Regione", "Comune", "Regione"]].rename(columns={"OBS_VALUE":"Morti"}).reset_index(drop=True)

In [51]:
df_merged = df_acc.merge(df_inj, on=["REF_AREA","TIME_PERIOD"], how="outer") # outer join to ensure no loss in data points
df_merged = df_merged.merge(df_dead, on=["REF_AREA","TIME_PERIOD"], how="outer")
df_merged = df_merged[["REF_AREA","Codice_Regione","Comune","Regione", "TIME_PERIOD", "Incidenti", "Feriti", "Morti"]]
df_merged = df_merged.rename(columns={
    "REF_AREA" : "Codice_Comune",
    "TIME_PERIOD" : "Anno"
})

In [52]:
df_merged

,Codice_Comune,Codice_Regione,Comune,Regione,Anno,Incidenti,Feriti,Morti
0,1001,1,Agliè,Piemonte,2014,6,11,0
1,1001,1,Agliè,Piemonte,2015,5,9,0
2,1001,1,Agliè,Piemonte,2016,5,8,0
3,1001,1,Agliè,Piemonte,2017,0,0,0
4,1001,1,Agliè,Piemonte,2018,1,1,0
...,...,...,...,...,...,...,...,...
85900,111107,20,Villaspeciosa,Sardegna,2020,2,3,0
85901,111107,20,Villaspeciosa,Sardegna,2021,5,7,0
85902,111107,20,Villaspeciosa,Sardegna,2022,1,1,0
85903,111107,20,Villaspeciosa,Sardegna,2023,4,5,0


In [54]:
#we can now enrich the dataset, computing the accident fatality index and injury index
df_merged["Indice Fatalità"] = (df_merged["Morti"]/df_merged["Incidenti"]).fillna(0).round(2)
df_merged["Indice Lesività"] = (df_merged["Feriti"]/df_merged["Incidenti"]).fillna(0).round(2)


In [57]:
df_historical = df_merged.copy()
df_historical.to_csv("data/historical_accidents.csv", index=False)

To ensure statistical rigor and avoid temporal bias (since demographic and automotive metrics are static data), we will partition this dataset in other two dataset, grouped by municipality and region.

In [104]:
df_municipalities_aggregated = df_merged.groupby("Codice_Comune")[["Incidenti","Feriti", "Morti"]].mean().reset_index()
df_municipalities_aggregated = df_municipalities_aggregated.rename(columns={
    "Incidenti":"Media Incidenti",
    "Feriti":"Media Feriti",
    "Morti":"Media Morti"
})
df_municipalities_merged = df_municipalities_aggregated.merge(df_municipalities, on="Codice_Comune", how="left")

In [105]:
df_municipalities


,Codice_Regione,Codice_Comune,Comune,Popolazione_Legale,Superficie(Kmq),Regione
0,1,1001,Agliè,2562,13.1463,Piemonte
1,1,1002,Airasca,3660,15.7393,Piemonte
2,1,1003,Ala di Stura,467,46.3316,Piemonte
3,1,1004,Albiano d'Ivrea,1637,11.7397,Piemonte
4,1,1006,Almese,6331,17.8741,Piemonte
...,...,...,...,...,...,...
7891,20,111103,Villaputzu,4509,181.3947,Sardegna
7892,20,111104,Villasalto,989,130.3596,Sardegna
7893,20,111105,Villasimius,3705,58.1781,Sardegna
7894,20,111106,Villasor,6599,86.8137,Sardegna


In [106]:
df_municipalities_merged = df_municipalities_merged[["Comune", "Popolazione_Legale", "Superficie(Kmq)", "Media Incidenti", "Media Feriti", "Media Morti"]]

In [107]:
df_municipalities_merged

,Comune,Popolazione_Legale,Superficie(Kmq),Media Incidenti,Media Feriti,Media Morti
0,Agliè,2562.0,13.1463,3.090909,5.090909,0.090909
1,Airasca,3660.0,15.7393,7.363636,12.545455,0.181818
2,Ala di Stura,467.0,46.3316,0.454545,0.545455,0.000000
3,Albiano d'Ivrea,1637.0,11.7397,4.000000,6.545455,0.181818
4,NaN,NaN,NaN,1.000000,2.200000,0.000000
...,...,...,...,...,...,...
8311,Villaputzu,4509.0,181.3947,6.000000,8.750000,0.625000
8312,Villasalto,989.0,130.3596,2.250000,3.250000,0.125000
8313,Villasimius,3705.0,58.1781,6.750000,9.250000,0.625000
8314,Villasor,6599.0,86.8137,13.000000,19.625000,1.375000


In [108]:
# We'll drop the rows for dissolved municipalities
df_municipalities_merged = df_municipalities_merged.dropna(subset=['Comune']).reset_index(drop=True)
df_municipalities_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 7896 entries, 0 to 7895
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Comune              7896 non-null   str    
 1   Popolazione_Legale  7896 non-null   float64
 2   Superficie(Kmq)     7896 non-null   float64
 3   Media Incidenti     7896 non-null   float64
 4   Media Feriti        7896 non-null   float64
 5   Media Morti         7896 non-null   float64
dtypes: float64(5), str(1)
memory usage: 370.3 KB


In [111]:
#We can now compute some metrics to enhance our future analysis
df_municipality_indicators = df_municipalities_merged.copy()


In [ ]:
#Accidents per 1,000 inhabitants (Per capita accident rate)
df_municipality_indicators["Incidenti_Pro_Capite_1000"] = ((df_municipality_indicators["Media Incidenti"]/df_municipality_indicators["Popolazione_Legale"])*1000).fillna(0)
#Total casualties (Fatalities + Injured) per 1,000 inhabitants (Per capita casualty rate)
df_municipality_indicators["Lesività_Pro_Capite_1000"] = (((df_municipality_indicators["Media Feriti"] + df_municipality_indicators["Media Morti"]) /df_municipality_indicators["Popolazione_Legale"])*1000).fillna(0)
#Average accidents per square kilometer (Accident density)
df_municipality_indicators["Densità_Incidenti_Kmq"] = (df_municipality_indicators["Media Incidenti"]/df_municipality_indicators["Superficie(Kmq)"]).fillna(0)

In [115]:
df_municipality_indicators.to_csv("data/municipality_indicators.csv", index=False)

We'll now group the dataset by Region to compute large-scale geographic indicators

In [121]:
df_regional = df_merged.groupby("Regione")[["Incidenti", "Feriti", "Morti"]].sum().reset_index()

In [122]:
df_municipalities_region = df_municipalities.groupby("Regione")[["Popolazione_Legale", "Superficie(Kmq)"]].sum().reset_index()

In [130]:
df_regional_comp = df_regional.merge(df_municipalities_region, on="Regione", how="left").dropna().reset_index(drop=True)

In [133]:
#regional name correction 
regioni = {
    'Trentino A.A.': 'Trentino-Alto Adige/Südtirol',
    'Friuli V.G.': 'Friuli-Venezia Giulia',
    'Emilia Romagna': 'Emilia-Romagna',
    "Valle D'Aosta": "Valle d'Aosta/Vallée d'Aoste"
}
df_aci["Regione"] = df_aci["Regione"].str.replace(regioni)

In [135]:
df_regional_fleet = df_regional_comp.merge(df_aci, on="Regione", how="left")

In [136]:
df_regional_fleet

,Regione,Incidenti,Feriti,Morti,Popolazione_Legale,Superficie(Kmq),Parco_Auto_2024
0,Abruzzo,33053,48021,816,1275950.0,10831.7158,923548
1,Basilicata,9991,15902,392,541168.0,10073.1768,391305
2,Calabria,29807,48183,1043,1855454.0,15219.2985,1376514
3,Campania,104813,153975,2456,5624420.0,13675.6929,3725175
4,Emilia-Romagna,179767,238923,3355,4425366.0,22501.8136,3082830
5,Friuli-Venezia Giulia,35389,46515,787,1194647.0,7933.2959,828929
6,Lazio,210098,284745,3630,5714882.0,17239.3387,4000575
7,Liguria,86345,107419,777,1509227.0,5417.6165,850098
8,Lombardia,329763,444195,4531,9943004.0,23863.0919,6426326
9,Marche,55108,76643,975,1487150.0,9346.0194,1063668


In [137]:
#We can now compute regional metrics to analyze the impact of population,territory size, and vehicle fleet density.
df_regional_fleet['Veicoli_Per_Kmq'] = (df_regional_fleet['Parco_Auto_2024'] / df_regional_fleet['Superficie(Kmq)']) #Vehicle density
df_regional_fleet['Motorizzazione_Pro_Capite_1000'] = ((df_regional_fleet['Parco_Auto_2024'] / df_regional_fleet['Popolazione_Legale']) * 1000)
df_regional_fleet['Rapporto Incidenti/Veicoli'] = ((df_regional_fleet['Incidenti'] / df_regional_fleet['Parco_Auto_2024']) * 1000) #incident to vehicle ratio per 1k vehicles

In [139]:
df_regional_fleet.to_csv("data/regiona_fleet_metrics.csv",index=False)